In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


In [0]:
-- 1
INSERT INTO ecommerce_orders VALUES
(109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

-- 2
INSERT INTO ecommerce_orders VALUES
(110,'Arjun Nair','Kochi','Mobile',1,30000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

-- 3
INSERT INTO ecommerce_orders VALUES
(112,'Neha Singh','Delhi','Camera',1,50000,'Shipped');

-- 4
INSERT INTO ecommerce_orders VALUES
(113,'Ravi Kumar','Chennai','Headphones',5,2000,'Placed');

-- 5
INSERT INTO ecommerce_orders VALUES
(114,'Suresh','Hyderabad','Mobile',1,25000,'Placed'),
(115,'Mahesh','Hyderabad','Tablet',2,26000,'Placed'),
(116,'Ramesh','Hyderabad','Laptop',1,77000,'Placed');

num_affected_rows,num_inserted_rows
3,3


In [0]:
-- 1
UPDATE ecommerce_orders
SET order_status = 'Shipped'
WHERE order_id = 101;

-- 2
UPDATE ecommerce_orders
SET quantity = quantity + 1
WHERE order_id = 102;

-- 3
UPDATE ecommerce_orders
SET price = 78000
WHERE product = 'Laptop';

-- 4
UPDATE ecommerce_orders
SET city = 'Secunderabad'
WHERE customer_name = 'Amit Sharma';

-- 5
UPDATE ecommerce_orders
SET order_status = 'Delivered'
WHERE product = 'Mobile';

-- 6
UPDATE ecommerce_orders
SET price = 2500
WHERE product = 'Headphones';

-- 7
UPDATE ecommerce_orders
SET price = price + 1000
WHERE product = 'Tablet';

-- 8
UPDATE ecommerce_orders
SET order_status = 'Processing'
WHERE city = 'Bangalore';

-- 9
UPDATE ecommerce_orders
SET quantity = 2
WHERE quantity = 1;

-- 10
UPDATE ecommerce_orders
SET city = 'Surat'
WHERE city = 'Ahmedabad';

num_affected_rows
1


In [0]:
-- 1
DELETE FROM ecommerce_orders
WHERE order_status = 'Cancelled';

-- 2
DELETE FROM ecommerce_orders
WHERE quantity > 3;

-- 3
DELETE FROM ecommerce_orders
WHERE product = 'Camera';

-- 4
DELETE FROM ecommerce_orders
WHERE city = 'Kolkata';

-- 5
DELETE FROM ecommerce_orders
WHERE price < 5000;

-- 6
DELETE FROM ecommerce_orders
WHERE customer_name LIKE 'A%';

-- 7
DELETE FROM ecommerce_orders
WHERE product = 'Tablet';

-- 8
DELETE FROM ecommerce_orders
WHERE city = 'Mumbai' AND quantity = 1;

-- 9
DELETE FROM ecommerce_orders
WHERE price > 80000;

-- 10 (last inserted)
DELETE FROM ecommerce_orders
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
-- 4
SELECT * FROM ecommerce_orders ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
114,Suresh,Hyderabad,Mobile,2,25000,Delivered


In [0]:
-- 5 Identify inserted vs updated
SELECT *,
CASE 
WHEN order_id IN (SELECT order_id FROM incoming_orders) THEN 'UPDATED/INSERTED'
ELSE 'EXISTING'
END AS status
FROM ecommerce_orders;

order_id,customer_name,city,product,quantity,price,order_status,status
108,Meera Nair,Bangalore,Laptop,2,78000,Processing,EXISTING
114,Suresh,Hyderabad,Mobile,2,25000,Delivered,EXISTING
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered,UPDATED/INSERTED
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped,UPDATED/INSERTED
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed,UPDATED/INSERTED
110,Divya Menon,Kochi,Mobile,1,27000,Placed,UPDATED/INSERTED
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed,UPDATED/INSERTED


In [0]:
-- 6 Update only order_status
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id
WHEN MATCHED THEN
UPDATE SET t.order_status = s.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
-- 7 Update quantity only if increased
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id
WHEN MATCHED AND s.quantity > t.quantity THEN
UPDATE SET t.quantity = s.quantity;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
-- 8 Ignore cancelled
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id
WHEN MATCHED AND s.order_status != 'Cancelled' THEN
UPDATE SET *
WHEN NOT MATCHED AND s.order_status != 'Cancelled' THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
-- 9 Insert only Laptop
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id
WHEN NOT MATCHED AND s.product = 'Laptop' THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
-- 10 Update price if higher
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id
WHEN MATCHED AND s.price > t.price THEN
UPDATE SET t.price = s.price;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
-- 1,2,3
MERGE INTO ecommerce_orders t
USING daily_updates s
ON t.order_id = s.order_id

WHEN MATCHED THEN
UPDATE SET t.quantity = s.quantity

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
-- 4 Count updated
SELECT COUNT(*) AS updated_count
FROM ecommerce_orders t
JOIN daily_updates s
ON t.order_id = s.order_id;

updated_count
3


In [0]:
-- 5 Count inserted
SELECT COUNT(*) AS inserted_count
FROM daily_updates s
LEFT JOIN ecommerce_orders t
ON s.order_id = t.order_id
WHERE t.order_id IS NULL;

inserted_count
0


In [0]:
-- 6 Final sorted table
SELECT * 
FROM ecommerce_orders
ORDER BY price DESC;


order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
114,Suresh,Hyderabad,Mobile,2,25000,Delivered
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
